### INITIALIZATION: 
IMPORT TOOLS, SET SEED, AND CREATE INDEPENDENT TABLES

In [1]:
# libraries
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

# seed
SEED = 42
np.random.seed(SEED)


print("Libraries loaded and seed set.")

Libraries loaded and seed set.


### Independent Entities

In [2]:
# Step 2: Independent Entities (Departments)
departments_data = [
    {"name": "Traffic & Transport", "daily_capacity": 50, "vulnerability_to_storm": 5.0, "base_rate": 20},
    {"name": "Public Works", "daily_capacity": 40, "vulnerability_to_storm": 8.0, "base_rate": 15},
    {"name": "Parks & Recreation", "daily_capacity": 10, "vulnerability_to_storm": 1.5, "base_rate": 2},
    {"name": "Animal Control", "daily_capacity": 15, "vulnerability_to_storm": 1.0, "base_rate": 5},
    {"name": "Sanitation", "daily_capacity": 60, "vulnerability_to_storm": 3.0, "base_rate": 45}
]

df_departments = pd.DataFrame(departments_data)

# Give each department a unique UUID
df_departments['department_id'] = [str(uuid.uuid4()) for _ in range(len(df_departments))]

print(df_departments.head())

                  name  ...                         department_id
0  Traffic & Transport  ...  f878c357-d9af-404a-8e3f-3cb73e8e6264
1         Public Works  ...  f56071f8-bddf-4096-beae-64cdd9f18e49
2   Parks & Recreation  ...  4c836cb1-533b-492e-bc37-87dd0883e73b
3       Animal Control  ...  8fc9877d-7ca6-4acc-a7a6-b621628e7fd4
4           Sanitation  ...  771eca92-56cd-4154-8241-cc41b15e7227

[5 rows x 5 columns]


In [3]:
# setup 
NUM_CITIZENS = 10000
BASE_DATE = datetime(2026, 8, 1)

# citizen ids
# generate 10000 ids for each citizen
citizen_ids = [str(uuid.uuid4()) for _ in range(NUM_CITIZENS)]

# dates
# generate random dates
random_days_ago = np.random.randint(1, 365,size=NUM_CITIZENS)
join_dates = [
    (BASE_DATE - timedelta(days=int(days))).strftime("%Y-%m-%d") 
    for days in random_days_ago
]

# clip L_civic
# np.random.normal creates the raw data without clipping 
# np.clip forces any number below 0.1 to become 0.1, and any above 2 to become 2.
raw_scores = np.random.normal(loc=0.5, scale=0.3, size=NUM_CITIZENS)
L_civics = np.clip(raw_scores, a_min=0.1, a_max=2.0)


# creation of data frame
df_citizens = pd.DataFrame({
    "citizen_id": citizen_ids,
    "join_date": join_dates,
    "L_civic": L_civics,
})

print(df_citizens.head())
print("\nL_civic:")
print(df_citizens["L_civic"].describe())

                             citizen_id   join_date   L_civic
0  57e43cdf-75e7-4404-b66a-33b0bea0e6c0  2026-04-20  0.798459
1  009a8433-3b98-4099-8f21-45f4166ac314  2025-08-17  0.471951
2  837a6427-15bb-4203-9af7-86aaf301286a  2025-11-03  1.152113
3  a7993b7f-3aad-402c-bc76-0a649cda950a  2026-04-16  0.100000
4  5a4482ba-545f-4048-8661-28a878fa6458  2026-05-21  0.211590

L_civic:
count    10000.000000
mean         0.511578
std          0.275631
min          0.100000
25%          0.296869
50%          0.499840
75%          0.701194
max          1.525584
Name: L_civic, dtype: float64


### Step 3B: Timeline & Latent Weather Shock
Simulate a 30-day calendar starting from  (August 1, 2026) and inject a hidden omitted variable  representing a decaying typhoon shock.

In [4]:
# Step 3B: Timeline & Latent Weather Shock

# 1. 30 consecutive days starting from BASE_DATE using timedelta list comprehension
# create a list of all dates from the starting base date
timeline_dates = [
    (BASE_DATE + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(30)
]

# 2. Vectorized initialization of latent storm shock (L_storm)
# intialize a 30 row list with 0.0
L_storm = np.zeros(30)

# 3. Inject decaying typhoon shock at specific indexes (Day 15, Day 16, Day 17)
# change the values to simulate a typoon situation
L_storm[14] = 1.0  # Index 14 (Day 15): Peak typhoon shock
L_storm[15] = 0.6  # Index 15 (Day 16): Receding floodwaters
L_storm[16] = 0.2  # Index 16 (Day 17): Residual shock

# Create timeline DataFrame
df_timeline = pd.DataFrame({
    "date": timeline_dates,
    "L_storm": L_storm
})

# Verify storm pulse injection
print("Timeline created. Storm pulse slice (index 13:18):")
print(df_timeline.iloc[13:18])

Timeline created. Storm pulse slice (index 13:18):
          date  L_storm
13  2026-08-14      0.0
14  2026-08-15      1.0
15  2026-08-16      0.6
16  2026-08-17      0.2
17  2026-08-18      0.0


### Step 3C: Timeline & Department Vulnerability Interaction Grid
Perform a Cartesian cross-join between `df_timeline` (30 days) and `df_departments` (5 departments) to create a 150-row simulation grid (`df_grid`).

In [5]:
# Step 3C: Timeline & Department Vulnerability Interaction Grid

# Perform Cartesian cross-join between timeline and departments
# merge the timeline with departments table, creating 150 rows
df_grid = df_timeline.merge(df_departments, how="cross")

# Verification
print(f"Simulation grid created with shape: {df_grid.shape}")
print("\nSample rows during storm peak (2026-08-15):")
print(df_grid[df_grid["date"] == "2026-08-15"][["date", "name", "L_storm", "vulnerability_to_storm", "base_rate"]])


Simulation grid created with shape: (150, 7)

Sample rows during storm peak (2026-08-15):
          date                 name  L_storm  vulnerability_to_storm  base_rate
70  2026-08-15  Traffic & Transport      1.0                     5.0         20
71  2026-08-15         Public Works      1.0                     8.0         15
72  2026-08-15   Parks & Recreation      1.0                     1.5          2
73  2026-08-15       Animal Control      1.0                     1.0          5
74  2026-08-15           Sanitation      1.0                     3.0         45


### Step 4: Core Ticket Generation Engine
Calculates expected daily ticket volume ($\lambda$) per department using $\lambda = \text{base\_rate} + (\text{base\_rate} \times L_{\text{storm}} \times \text{vulnerability\_to\_storm})$. Samples realized ticket counts using Poisson distribution (`np.random.poisson(lam)`) and assigns tickets to citizens weighted by their civic engagement score $L_{\text{civic}}$ (hyper-reporters).

In [6]:
# Step 4: Core Ticket Generation Engine

# 1. Expected daily ticket rate (lambda)
df_grid["lambda"] = df_grid["base_rate"] + (
    df_grid["base_rate"] * df_grid["L_storm"] * df_grid["vulnerability_to_storm"]
)

# 2. Sample realized ticket count per grid row using Poisson distribution
df_grid["num_tickets"] = np.random.poisson(df_grid["lambda"])

# 3. Weighted random choice probability for citizens (hyper-reporters)
citizen_weights = df_citizens["L_civic"].values / df_citizens["L_civic"].sum()

# 4. Generate df_tickets records
ticket_records = []
for idx, row in df_grid.iterrows():
    count = row["num_tickets"]
    if count > 0:
        assigned_citizens = np.random.choice(
            df_citizens["citizen_id"].values,
            size=count,
            p=citizen_weights
        )
        for c_id in assigned_citizens:
            ticket_records.append({
                "ticket_id": str(uuid.uuid4()),
                "created_at": row["date"],
                "department_id": row["department_id"],
                "citizen_id": c_id
            })

df_tickets = pd.DataFrame(ticket_records)

# Verification & Summaries
print(f"Total synthetic tickets generated: {len(df_tickets)}")
print("\nHead of df_tickets:")
print(df_tickets.head())
print("\nDaily ticket counts during storm surge (2026-08-13 to 2026-08-18):")
daily_summary = df_grid.groupby("date")["num_tickets"].sum().reset_index()
print(daily_summary.iloc[12:18])


Total synthetic tickets generated: 3199

Head of df_tickets:
                              ticket_id  ...                            citizen_id
0  ab4abaec-82b8-4005-b765-2e444e307e61  ...  df69ae53-c3ed-4efd-b242-730a5ba95d7a
1  bc03839b-8cbd-4f75-8fd5-3ea1f918bc8b  ...  4a66e41e-6dd1-4d9e-a033-928aa5349ce9
2  607653a1-89c4-41ef-a8bd-de5a349d257b  ...  be60fdd0-384c-4c6c-b3cf-9d47acbbc6f1
3  72b2571f-1c64-4980-b229-c7541cdf7312  ...  9e6e4d11-4332-4383-bf2d-d12822ec93fd
4  574ee7ca-7948-4e43-b4cf-ec772e94b97a  ...  83761375-2e89-4b25-a351-e68f16518403

[5 rows x 4 columns]

Daily ticket counts during storm surge (2026-08-13 to 2026-08-18):
          date  num_tickets
12  2026-08-13           87
13  2026-08-14           65
14  2026-08-15          491
15  2026-08-16          275
16  2026-08-17          141
17  2026-08-18           86
